# Notebook 3: PV-Potenzial – Adress- und Gemeindenvergleich Schweiz

Dieses Notebook kombiniert die PVOUT-Solardaten (aus Notebook 1) mit den Stromtarifen (aus Notebook 2)
und berechnet den **PV-Attraktivitätsscore** für jede Schweizer Gemeinde.

**Formel:** `Score = PVOUT [kWh/kWp/Jahr] × Stromtarif [Rp./kWh]`  
**Ergebnis:** Rp/kWp/Jahr – wie viel Strom pro installierter Leistungseinheit jährlich eingespart wird.

Beispiel: Score 14 000 Rp/kWp/Jahr → 5-kWp-Anlage spart ca. **700 CHF/Jahr**.

**Warum Score = PVOUT × Tarif?**
- PVOUT = kWh/kWp/Jahr (wie viel Strom eine Anlage produziert)
- Stromtarif = Rp./kWh (wie viel der Strom kostet)
- Score = Rp/kWp/Jahr (wie viel man durch Eigenverbrauch spart)

Wichtig: Eine sonnige Gemeinde mit **günstigem Strom** kann einen tieferen Score haben als eine
weniger sonnige Gemeinde mit **teurem Strom** – das ist korrekt! PV lohnt sich dort mehr, wo Strom teuer ist.

**Input-Daten:**
- `pvout_gemeinden.csv` – mittlerer PVOUT pro Gemeinde (aus Notebook 1)
- `df_h4_processed.csv` – Stromtarife H4-Haushalt pro Gemeinde (aus Notebook 2)
- `PVOUT.tif` – Solardaten-Raster für adressgenaue Pixelabfrage
- `swissBOUNDARIES3D.gpkg` – Gemeindegrenzen für Karte und Spatial Join

---

## Schritt 1: Bibliotheken installieren

**Ziel:** Alle benötigten Python-Pakete laden. Die Quelldateien werden direkt vom **GitHub-Repository** geladen.

**Bibliotheken und ihre Aufgaben in diesem Notebook:**

| Bibliothek | Wird verwendet in | Aufgabe |
|---|---|---|
| `pandas` | Schritt 3, 4, 11 | Tabellen laden, Daten zusammenführen (Join), Ranking erstellen |
| `rasterio` | Schritt 8 | PVOUT.tif öffnen und Pixelwert an bestimmten Koordinaten ablesen |
| `geopandas` | Schritt 5, 9, 12 | Gemeindegrenzen laden, Koordinatensystem umrechnen, Karten zeichnen |
| `shapely` | Schritt 9 | Punkt-Objekt aus Koordinaten erstellen (für Spatial Join benötigt) |
| `folium` | Schritt 12 | Interaktive Karte mit Choropleth und Adress-Marker erstellen |
| `fiona` | Schritt 5 | Treiber zum Öffnen von `.gpkg`-Dateien (GeoPackage) |
| `requests` | Schritt 7 | HTTP-Anfrage an die swisstopo Geocoding-API |
| `geopy` | Schritt 7 | Geocoding-Fallback via Nominatim/OpenStreetMap |
| `ipywidgets` | Schritt 6 | Visuelles Adress-Eingabefeld |


In [1]:
!pip install rasterio geopandas folium fiona geopy ipywidgets --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.3 MB/s eta 0:00:00


In [2]:
# Pfade zu den Quelldateien im GitHub-Repository
PVOUT_PATH     = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/PVOUT.tif'
GPKG_PATH      = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg'
PVOUT_CSV_PATH = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/pvout_gemeinden.csv'
TARIFE_PATH    = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/df_h4_processed.csv'

print(f'PVOUT.tif:            {PVOUT_PATH}')
print(f'GPKG:                 {GPKG_PATH}')
print(f'pvout_gemeinden.csv:  {PVOUT_CSV_PATH}')
print(f'df_h4_processed.csv:  {TARIFE_PATH}')


PVOUT.tif:            https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/PVOUT.tif
GPKG:                 https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg
pvout_gemeinden.csv:  https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/pvout_gemeinden.csv
df_h4_processed.csv:  https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/df_h4_processed.csv


## Schritt 2: Dateien laden

**Ziel:** Alle vier benötigten Dateien direkt vom GitHub-Repository laden.

**Was sind diese Dateien?**

**pvout_gemeinden.csv** – Output aus Notebook 1  
Enthält den mittleren PVOUT-Wert (kWh/kWp/Jahr) für jede der 2132 Schweizer Gemeinden, berechnet aus dem Global Solar Atlas.

**df_h4_processed.csv** – Output aus Notebook 2  
Enthält den H4-Haushaltsstromtarif (Rp./kWh) für ca. 2100 Gemeinden aus den ElCom-Daten.
Bei Gemeinden mit mehreren Netzbetreibern wird der höchste Tarif verwendet.

**PVOUT.tif** – Solardaten-Raster (Global Solar Atlas v2)  
Wird für die adressgenaue Pixelabfrage in Schritt 8 benötigt. Auflösung: ca. 1 km × 1 km.

**swissBOUNDARIES3D.gpkg** – Gemeindegrenzen (swisstopo)  
Polygone aller Schweizer Gemeinden. Werden für den Spatial Join (Schritt 9) und die Choropleth-Karte (Schritt 12) benötigt.

**Was der Code macht:**
- Alle Dateien werden via HTTPS direkt von GitHub geladen
- Das GPKG wird lokal zwischengespeichert, da SQLite-Dateien nicht direkt per URL geöffnet werden können


In [3]:
import pandas as pd

url = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/Rohdaten-Tarife-Standard-Produkt.csv'

# Lese die CSV-Datei in ein pandas DataFrame
df = pd.read_csv(url)

# Zeige die ersten fünf Zeilen des DataFrames an
df.head()

,uid,netzbetreiber,kategorieName,gemeindeNummer,gemeindeName,kanton,total,energie,abgaben,netznutzung,netzzuschlag,messwesen
0,CHE-105.981.944,AEW Energie AG,C1,4172,Münchwilen (AG),AG,26.594,11.4,0.689,11.53,2.3,54.0
1,CHE-105.981.944,AEW Energie AG,C1,4253,Magden,AG,26.594,11.4,0.689,11.53,2.3,54.0
2,CHE-105.981.944,AEW Energie AG,C1,4106,Mönthal,AG,26.555,11.4,0.650,11.53,2.3,54.0
3,CHE-105.981.944,AEW Energie AG,C1,4068,Hägglingen,AG,26.594,11.4,0.689,11.53,2.3,54.0
4,CHE-105.981.944,AEW Energie AG,C1,4258,Rheinfelden,AG,26.594,11.4,0.689,11.53,2.3,54.0


## Schritt 3: Daten laden und Überblick verschaffen

**Ziel:** Beide CSV-Dateien in pandas DataFrames laden.

**Was sind die Spalten?**

**pvout_gemeinden.csv:**

| Spalte | Typ | Beschreibung |
|---|---|---|
| `gemeinde_nr` | int | BFS-Gemeindenummer (eindeutiger Schlüssel für den Join) |
| `gemeinde_name` | str | Offizieller Gemeindename |
| `kanton_name` | str | Deutscher Kantonsname |
| `pvout_kwh_kwp_year` | float | Mittlerer PVOUT-Wert der Gemeinde [kWh/kWp/Jahr] |

**df_h4_processed.csv:**

| Spalte | Typ | Beschreibung |
|---|---|---|
| `netzbetreiber` | str | Name des Stromnetzbetreibers |
| `gemeindeNummer` | int | BFS-Gemeindenummer (Join-Schlüssel) |
| `gemeindeName` | str | Gemeindename |
| `kanton` | str | Kantonskürzel (z.B. BE, ZH) |
| `total` | float | Gesamtstromtarif H4 [Rp./kWh] – inkl. Energie, Netz, Abgaben |

**Join-Schlüssel:** `pvout_gemeinden.gemeinde_nr` ↔ `df_h4_processed.gemeindeNummer` (beide int, BFS-Nummer)

In [4]:
pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

print('=== pvout_gemeinden.csv ===')
print(f'Zeilen: {len(pvout)}, Spalten: {list(pvout.columns)}')
print(pvout.head(3).to_string(index=False))

print()
print('=== df_h4_processed.csv ===')
print(f'Zeilen: {len(tarife)}, Spalten: {list(tarife.columns)}')
print(tarife.head(3).to_string(index=False))

=== pvout_gemeinden.csv ===
Zeilen: 2119, Spalten: ['gemeinde_nr', 'gemeinde_name', 'kanton_name', 'pvout_kwh_kwp_year']
 gemeinde_nr gemeinde_name kanton_name  pvout_kwh_kwp_year
         131      Adliswil      Zürich         1155.791391
        3714     Rheinwald  Graubünden         1273.649336
        5722         Grens        Vaud         1292.010905

=== df_h4_processed.csv ===
Zeilen: 2123, Spalten: ['netzbetreiber', 'gemeindeNummer', 'gemeindeName', 'kanton', 'total']
                              netzbetreiber  gemeindeNummer       gemeindeName kanton  total
Elektrizitätswerke des Kantons Zürich (EKZ)               1    Aeugst am Albis     ZH 24.136
Elektrizitätswerke des Kantons Zürich (EKZ)               2 Affoltern am Albis     ZH 24.136
Elektrizitätswerke des Kantons Zürich (EKZ)               3         Bonstetten     ZH 24.136


## Schritt 4: Datenqualität prüfen (Qualitätskontrolle 1)

**Ziel:** Sicherstellen, dass die Daten vollständig und plausibel sind, bevor wir mit der Analyse beginnen.

**Warum haben die beiden Datensätze unterschiedlich viele Zeilen?**
- `pvout_gemeinden.csv` hat 2132 Gemeinden (alle Gemeinden mit PVOUT-Daten aus Notebook 1)
- `df_h4_processed.csv` hat ca. 2100 Gemeinden (aus ElCom-Daten)

Nicht jede Gemeinde ist in beiden Datensätzen vorhanden. Mögliche Gründe:
- Gemeindefusionen seit dem Erhebungsjahr der ElCom-Daten (BFS-Nummern haben sich geändert)
- Kleine Gemeinden ohne eigenen Netzbetreiber in den ElCom-Daten
- Fehlende oder unvollständige Meldungen einzelner Netzbetreiber

**Was passiert mit den fehlenden Gemeinden?**

Für das nationale Ranking (Schritt 11) werden nur Gemeinden verwendet, die in **beiden** Datensätzen vorhanden sind.
Das nennt man einen *Inner Join*. Gemeinden ohne Tarif-Daten erscheinen grau in der Karte und nicht im Ranking.

Für die **Adress-Analyse** (Schritt 10) gibt es einen Fallback: wenn keine Tarif-Daten gefunden werden,
wird der Schweizer Durchschnittstarif verwendet – mit einem entsprechenden Hinweis.

**Was der Code macht:**
- Zeigt die Anzahl Gemeinden in beiden Datensätzen
- Listet Gemeinden aus PVOUT, für die kein Tarif gefunden wird
- Prüft den Wertebereich beider Datensätze auf Plausibilität

In [5]:
print(f'PVOUT-Gemeinden      : {len(pvout)}')
print(f'Tarif-Gemeinden      : {len(tarife)}')

ohne_tarif = pvout[~pvout['gemeinde_nr'].isin(tarife['gemeindeNummer'])]
print(f'Ohne Tarif-Daten     : {len(ohne_tarif)} Gemeinden')
if len(ohne_tarif) > 0:
    print(ohne_tarif[['gemeinde_name', 'kanton_name']].head(10).to_string(index=False))

tarif_min = tarife['total'].min()
tarif_max = tarife['total'].max()
pvout_min = pvout['pvout_kwh_kwp_year'].min()
pvout_max = pvout['pvout_kwh_kwp_year'].max()

print()
print(f'Tarif-Bereich        : {tarif_min:.1f} - {tarif_max:.1f} Rp./kWh')
print(f'PVOUT-Bereich        : {pvout_min:.0f} - {pvout_max:.0f} kWh/kWp/Jahr')
print(f'Score-Bereich (grob) : {pvout_min * tarif_min:.0f} - {pvout_max * tarif_max:.0f} Rp/kWp/Jahr')

PVOUT-Gemeinden      : 2119
Tarif-Gemeinden      : 2123
Ohne Tarif-Daten     : 16 Gemeinden
                  gemeinde_name kanton_name
                 Zürichsee (ZH)      Zürich
          Lac de Neuchâtel (BE)        Bern
                     Greifensee      Zürich
                 Bielersee (BE)        Bern
                 Bielersee (NE)   Neuchâtel
Comunanza Cadenazzo/Monteceneri      Ticino
                  Bodensee (TG)     Thurgau
               Fétigny-Ménières    Fribourg
                    Brienzersee        Bern
                      Thunersee        Bern

Tarif-Bereich        : 9.6 - 43.6 Rp./kWh
PVOUT-Bereich        : 885 - 1485 kWh/kWp/Jahr
Score-Bereich (grob) : 8531 - 64777 Rp/kWp/Jahr


### Anzahl Gemeinden nur in einem Notebook

In [6]:
pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

# Gemeinden nur in pvout_gemeinden.csv
gemeinden_only_in_pvout = pvout[~pvout['gemeinde_nr'].isin(tarife['gemeindeNummer'])]

# Gemeinden nur in df_h4_processed.csv
gemeinden_only_in_tarife = tarife[~tarife['gemeindeNummer'].isin(pvout['gemeinde_nr'])]

print(f"Anzahl Gemeinden nur in pvout_gemeinden.csv: {len(gemeinden_only_in_pvout)}")
print(f"Anzahl Gemeinden nur in df_h4_processed.csv: {len(gemeinden_only_in_tarife)}")

Anzahl Gemeinden nur in pvout_gemeinden.csv: 16
Anzahl Gemeinden nur in df_h4_processed.csv: 20


### Gemeinden nur in pvout_gemeinden.csv

Auch bei den restlichen Gemeinden mit Kantonszuordnung gibt es jeweils einen administrativen oder zeitlichen Grund, weshalb sie in den Stromtarifdaten fehlen. Sie lassen sich in fünf Kategorien einteilen:
* **10 Seen**
  Möglicherweise damit die Gewässergrenzen der Seen in der Karte enthalten sind. Sie werden weiterhin auf der Karte angezeigt.
* **2 Gemeinschaftsareale ohne Bewohner**​

  5391 Comunanza Cadenazzo/Monteceneri (TI) und 2391 Staatswald Galm (FR) ein gemeindefreies Waldareal im Seebezirk FR, das keiner politischen Gemeinde zugeordnet ist​
* **2 neue Gemeinden sind durch Fusionen 2025/2026 entstanden​**

  5395 Lema (TI) - per 6. April 2025 aus Astano, Bedigliora, Curio, Miglieglia und Novaggio fusioniert (benannt nach dem Monte Lema).​

  2056 Fétigny-Ménières (FR) – per 1. Januar 2026 aus Fétigny und Ménières entstanden.​
* **1 Kantonswechsel per 01.01.2026**

  6831 Moutier (JU): von BE zu JU​

* **Keine bekannte Datenlücke**

  2822 Augst (BL): reguläre, bewohnte Gemeinde ohne erkennbaren Sonderstatus, vermutlich eine Lücke im Tarifdatensatz, die geprüft werden sollte.

In [7]:
pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

# Gemeinden nur in pvout_gemeinden.csv
gemeinden_only_in_pvout = pvout[~pvout['gemeinde_nr'].isin(tarife['gemeindeNummer'])]

print(gemeinden_only_in_pvout.to_string(index=False))

 gemeinde_nr                   gemeinde_name      kanton_name  pvout_kwh_kwp_year
        9051                  Zürichsee (ZH)           Zürich         1164.118056
        9152           Lac de Neuchâtel (BE)             Bern         1218.017212
        9040                      Greifensee           Zürich         1181.104492
        9149                  Bielersee (BE)             Bern         1203.667308
        9150                  Bielersee (NE)        Neuchâtel         1216.281982
        5391 Comunanza Cadenazzo/Monteceneri           Ticino         1203.352637
        9329                   Bodensee (TG)          Thurgau         1152.145197
        2056                Fétigny-Ménières         Fribourg         1279.862165
        9089                     Brienzersee             Bern         1122.164844
        9073                       Thunersee             Bern         1188.942264
        9328                   Bodensee (SG)       St. Gallen         1140.855925
        6831    

### Seen entfernen

In [8]:
# Lade pvout und tarife erneut, um den ursprünglichen Zustand wiederherzustellen
pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

# Identifiziere Gemeinden, die in pvout, aber nicht in tarife sind
ohne_tarif_original = pvout[~pvout['gemeinde_nr'].isin(tarife['gemeindeNummer'])]

# Filtere diese Gemeinden weiter, um diejenigen zu finden, die "Lac" oder "see" im Namen enthalten
# (?i) macht die Suche "case-insensitive"
gemeinden_zu_entfernen_mask = ohne_tarif_original['gemeinde_name'].str.contains('Lac|see', case=False, na=False)
entferne_gemeinde_nrs = ohne_tarif_original[gemeinden_zu_entfernen_mask]['gemeinde_nr']

# Zeige die Gemeinden an, die entfernt werden
if not entferne_gemeinde_nrs.empty:
    print("Folgende Gemeinden werden entfernt:")
    print(ohne_tarif_original[gemeinden_zu_entfernen_mask][['gemeinde_name', 'kanton_name', 'pvout_kwh_kwp_year']].to_string(index=False))
    print("\n")
else:
    print("Keine Gemeinden gefunden, die den Kriterien entsprechen und entfernt werden müssen.\n")

# Entferne diese Gemeinden aus dem originalen pvout DataFrame
initial_pvout_count = len(pvout)
pvout = pvout[~pvout['gemeinde_nr'].isin(entferne_gemeinde_nrs)]

print(f"Anzahl Gemeinden in pvout vor der Entfernung: {initial_pvout_count}")
print(f"Anzahl Gemeinden entfernt: {len(entferne_gemeinde_nrs)}")
print(f"Anzahl Gemeinden in pvout nach der Entfernung: {len(pvout)}")

Folgende Gemeinden werden entfernt:
        gemeinde_name kanton_name  pvout_kwh_kwp_year
       Zürichsee (ZH)      Zürich         1164.118056
Lac de Neuchâtel (BE)        Bern         1218.017212
           Greifensee      Zürich         1181.104492
       Bielersee (BE)        Bern         1203.667308
       Bielersee (NE)   Neuchâtel         1216.281982
        Bodensee (TG)     Thurgau         1152.145197
          Brienzersee        Bern         1122.164844
            Thunersee        Bern         1188.942264
        Bodensee (SG)  St. Gallen         1140.855925
Lac de Neuchâtel (NE)   Neuchâtel         1204.469296


Anzahl Gemeinden in pvout vor der Entfernung: 2119
Anzahl Gemeinden entfernt: 10
Anzahl Gemeinden in pvout nach der Entfernung: 2109


## Analyse Gemeinde Augst (BL)

Suche nach Gemeinden in df_h4_processed, die 'Augst' enthalten

In [9]:
# r'.*Augst.*' ist ein regulärer Ausdruck, der beliebige Zeichen davor oder danach erlaubt
augst_gemeinden_h4 = tarife[tarife['gemeindeName'].str.contains('Augst', case=False, na=False)]

if not augst_gemeinden_h4.empty:
    print("Gemeinden in df_h4_processed.csv, die 'Augst' enthalten:")
    print(augst_gemeinden_h4.to_string(index=False))
else:
    print("Keine Gemeinden mit 'Augst' im Namen in df_h4_processed.csv gefunden.")

Gemeinden in df_h4_processed.csv, die 'Augst' enthalten:
 netzbetreiber  gemeindeNummer gemeindeName kanton  total
AEW Energie AG            4252  Kaiseraugst     AG 27.892


#### Nach Gemeindenummer filtern

In [10]:
# Check für gemeindeNummer 2822 in df_h4_processed
result = df_h4_processed[df_h4_processed['gemeindeNummer'] == 2822]

if not result.empty:
    print("Eine Position mit der Gemeindenummer 2822 existiert in df_h4_processed:")
    print(result.to_string(index=False))
else:
    print("Keine Position mit der Gemeindenummer 2822 in df_h4_processed gefunden.")

NameError: name 'df_h4_processed' is not defined

#### df_h4_processed nach Kanton 'BL' filtern

Bei der einzelnen Durchsicht der Gemeinden des Kantons Basel-Landschaft konnte Augst nicht gefunden werden.

Als Gegencheck wurde die Website von [Elcom](https://www.strompreis.elcom.admin.ch/municipality/2822?period=2024&priceComponent=charge&category=H4&product=standard&view=collapsed&operator=$0) herangezogen. Es ist ersichtlich, dass auch 2025 die Daten dieser Gemeinde gefehlt haben. Bis 2024 waren entsprechende Daten hingegen vorhanden.

In [ ]:
# df_h4_processed nach Kanton 'BL' filtern
bl_gemeinden_h4 = df_h4_processed[df_h4_processed['kanton'] == 'BL']

if not bl_gemeinden_h4.empty:
    print("Gemeinden im Kanton Basel-Landschaft (BL) in df_h4_processed:")
    display(bl_gemeinden_h4[['gemeindeNummer', 'gemeindeName', 'kanton', 'total']])
else:
    print("Keine Gemeinden für Basel-Landschaft (BL) in df_h4_processed gefunden.")

#### Netzbetreiber Genossenschaft Elektra Baselland



Der Netzbetreiber Genossenschaft Elektra Baselland betreut 49 Gemeinden in der Kategorie H4. Da der Tarif für alle betreuten Gemeinden identisch ist, wird der Wert auch für die Gemeinde Augst ausgegeben, allerdings ohne Angabe des Netzbetreibers.

In [ ]:
elektra_baselland = tarife[tarife['netzbetreiber'].str.contains('Genossenschaft Elektra Baselland', case=False, na=False)]

if not elektra_baselland.empty:

    max_total = elektra_baselland['total'].max()
    min_total = elektra_baselland['total'].min()
    num_rows = len(elektra_baselland)

    print(f"\nAnzahl Zeilen für 'Genossenschaft Elektra Baselland': {num_rows}")
    print(f"Höchster Totalwert für 'Genossenschaft Elektra Baselland': {max_total:.3f}")
    print(f"Geringster Totalwert für 'Genossenschaft Elektra Baselland': {min_total:.3f}")
else:
    print("Keine Positionen für 'Genossenschaft Elektra Baselland' gefunden.")

#### Gemeinden Augst ergänzen

Dieser Code fügt eine neue Zeile für die Gemeinde 'Augst' (BFS 2822) mit spezifischen Daten, einschliesslich des Netzbetreibers 'Genossenschaft Elektra Baselland' und einem Vermerk zum Datenstand, zum tarife DataFrame hinzu. Eine plausible Begründung konnte nicht gefunden werden. Möglicherweise wurden die aktuellen Daten nicht nachgereicht oder es gab einen Netzbetreiberwechsel, der nicht gemeldet wurde.

In [ ]:
# Daten für Augst hinzufügen
new_row_augst = pd.DataFrame({
    'netzbetreiber': ['Genossenschaft Elektra Baselland (Stand 2024, keine aktuellen Daten)'],
    'gemeindeNummer': [2822],
    'gemeindeName': ['Augst'],
    'kanton': ['BL'],
    'total': [30.196]
})

# Die neue Zeile zum 'tarife' DataFrame hinzufügen
tarife = pd.concat([tarife, new_row_augst], ignore_index=True)

print("Gemeinde 'Augst' wurde zum 'tarife' DataFrame hinzugefügt (BFS 2822).")
print("Überprüfe den Eintrag:")
print(tarife[tarife['gemeindeNummer'] == 2822].to_string(index=False))

### Gemeinden nur in df_h4_processed.csv

Die folgenden 17 Positionen sind aufgelöste Gemeinden, die durch Fusion verschwunden sind:

  **LU: 1057 Honau** → fusioniert am 1.1.2025 mit Root zur
  Gemeinde Root. Honau war die kleinste Luzerner Gemeinde (~500 EW).

  **FR:** **2016 Fétigny und 2027 Ménières** → fusioniert am 1.1.2026 zur neuen Gemeinde Fétigny-Ménières (2056). Diese Gemeinde erschien im PVOUT-Datensatz neu.
  **2278 Ulmiz** → fusioniert am 1.1.2026 mit Gurmels.

  **SO: 2456 Lüterswil-Gächliwil** → fusioniert am 1.1.2024 mit der Gemeinde Buchegg.

  **GR:** **3932 Tschiertschen-Praden** → fusioniert am 1.1.2025 mit der Stadt Chur (nach den vorherigen Eingemeindungen von Maladers 2020 und Haldenstein 2021).

  **AG:** **4122 Villnachern** → fusioniert am 1.1.2026 mit Brugg (Brugg hatte bereits 2020 Schinznach-Bad eingemeindet).
  **4179 Ueken** → fusioniert bereits am 1.1.2023 mit Herznach zur neuen Gemeinde Herznach-Ueken. Auffällig: Ueken existiert seit 2023 nicht mehr – die Tarifdaten verwenden hier eine sehr veraltete BFS-Nummer.

  **TI - Region Lema (Malcantone): 5146 Astano, 5149 Bedigliora, 5181 Curio, 5200 Miglieglia, 5207 Novaggio** → alle fusioniert am 6.4.2025 zur neuen Gemeinde Lema (5395) – das ist die Gemeinde, die im PVOUT-Datensatz neu erscheint.

  **TI – Leventina: 5064 Bodio** → eingemeindet am 6.4.2025 in die Gemeinde Giornico.
  **5078 Prato (Leventina)** → fusioniert am 6.4.2025 mit der Nachbargemeinde Quinto.

  **NE:** **6454 Hauterive (NE)** → fusioniert am 1.1.2025 mit Enges, La Tène und Saint-Blaise zur neuen Gemeinde Laténa.

  **JU:** **6744 La Chaux-des-Breuleux** → fusioniert bereits am 1.1.2023 mit Les Breuleux. Mit nur ~96 Einwohnern und 4 km² war sie eine der kleinsten Jura-Gemeinden.

In [ ]:
pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

# Gemeinden nur in df_h4_processed.csv
gemeinden_only_in_tarife = tarife[~tarife['gemeindeNummer'].isin(pvout['gemeinde_nr'])]

print(gemeinden_only_in_tarife.to_string(index=False))

### Liste der (neuen) Gemeindenamen nach Fusion

Bei den fusionierten Gemeinden Lema und Fétigny-Ménières lautet der neue Gemeindename anders, daher wird der passende Datensatz nicht im pvout_gemeinden.csv gefunden. Da im Dataset erkennbar ist, dass die vorherigen Gemeinden dasselbe Total aufweisen, wird der Gemeindename eines Datasets korrigiert und der andere gelöscht.

Alle übrigen neuen Gemeindennamen sind in der df_h4_processed.csv enthalten. Daher werden diese Zeilen gelöscht.

In [ ]:
# Liste der (neuen) Gemeindenamen nach Fusionen
municipalities_to_check = [
    'Root',
    'Fétigny-Ménières',
    'Gurmels',
    'Buchegg',
    'Chur',
    'Brugg',
    'Herznach-Ueken',
    'Lema',
    'Giornico',
    'Quinto',
    'Laténa',
    'Les Breuleux',
    'Gravesano',
]

PVOUT_NAME_COL = 'gemeinde_name'
TARIFE_NAME_COL = 'gemeindeName'

print('Checking municipalities in pvout and tarife datasets:')
print('----------------------------------------------------')

for name in municipalities_to_check:
    in_pvout = name in pvout[PVOUT_NAME_COL].values
    in_tarife = name in tarife[TARIFE_NAME_COL].values
    status_pvout = '✓ vorhanden' if in_pvout else '✗ NICHT vorhanden'
    status_tarife = '✓ vorhanden' if in_tarife else '✗ NICHT vorhanden'
    print(f'{name}:\n  - In pvout_gemeinden.csv: {status_pvout}\n  - In df_h4_processed.csv: {status_tarife}\n')

### Anpassungen für fusionierte Gemeinden

In [ ]:
# 'Fétigny' zu 'Fétigny-Ménières' aktualisieren und die Gemeindenummer auf 2056 ändern
tarife.loc[tarife['gemeindeName'] == 'Fétigny', 'gemeindeName'] = 'Fétigny-Ménières'
tarife.loc[tarife['gemeindeName'] == 'Fétigny-Ménières', 'gemeindeNummer'] = 2056

# Die Zeile für 'Ménières' (Gemeindenummer 2027) entfernen
tarife = tarife[tarife['gemeindeNummer'] != 2027].copy()

print("df_h4_processed.csv (tarife) nach den Anpassungen:")
print(tarife[(tarife['gemeindeNummer'] == 2056) | (tarife['gemeindeNummer'] == 2027)].to_string(index=False))

In [ ]:
# 'Astano' zu 'Lema' aktualisieren und die Gemeindenummer auf 5395 ändern
tarife.loc[tarife['gemeindeNummer'] == 5146, 'gemeindeName'] = 'Lema'
tarife.loc[tarife['gemeindeNummer'] == 5146, 'gemeindeNummer'] = 5395

# Die Zeilen für die Gemeinden, die in Lema fusioniert sind, entfernen
municipalities_to_remove = [5149, 5181, 5200, 5207]
tarife = tarife[~tarife['gemeindeNummer'].isin(municipalities_to_remove)].copy()

print("df_h4_processed.csv (tarife) nach den Anpassungen für Lema:")
print(tarife[(tarife['gemeindeNummer'] == 5395) | tarife['gemeindeNummer'].isin([5146, 5149, 5181, 5200, 5207])].to_string(index=False))

In [ ]:
from IPython.display import display

# Vollständige Anzeige erzwingen (keine abgeschnittenen Tabellen)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Lade df_h4_processed aus der Originalquelle, um zu sehen, was entfernt WÜRDE
# 'tarife' ist bereits der bereinigte DataFrame
df_h4_processed_original = pd.read_csv(TARIFE_PATH)

print(f"Anzahl Zeilen 'df_h4_processed_original' vorher: {len(df_h4_processed_original)}\n")

# Spaltenname für die Gemeindenummer
h4_col = 'gemeindeNummer'

# Maske: Zeilen, deren Gemeinde NICHT in 'tarife' vorkommt (d.h. sie wurden in 'tarife' bereits gelöscht oder fusioniert)
mask_to_remove = ~df_h4_processed_original[h4_col].isin(tarife['gemeindeNummer'])

# Zu löschende Zeilen separat speichern (bleiben nach der Löschung verfügbar)
removed_rows = df_h4_processed_original[mask_to_remove].copy()

print(f"→ {len(removed_rows)} Zeile(n) würden aus der ursprünglichen 'df_h4_processed' gelöscht werden, um 'tarife' anzupassen:")
print("=" * 70)
display(removed_rows)
print("=" * 70)

# Die 'df_h4_processed' Variable wird nun direkt auf den bereits bereinigten 'tarife' DataFrame gesetzt.
# Dies ist effizienter, da 'tarife' bereits die gewünschte, bereinigte Version ist.
df_h4_processed = tarife.copy()

print(f"Effektiv 'entfernt' (im Sinne der Anpassung): {len(removed_rows)} Zeile(n)\n")
print(f"\nAnzahl Zeilen 'df_h4_processed' nach der Bereinigung (entspricht 'tarife'): {len(df_h4_processed)}")

# Gemeinden identifizieren, die nur im aktuellen df_h4_processed (tarife) erscheinen (nicht in pvout)
gemeinden_only_in_current_df_h4_processed = df_h4_processed[~df_h4_processed['gemeindeNummer'].isin(pvout['gemeinde_nr'])]

print(f"Anzahl Gemeinden nur im aktuellen df_h4_processed (tarife): {len(gemeinden_only_in_current_df_h4_processed)}")

if not gemeinden_only_in_current_df_h4_processed.empty:
    print("Folgende Gemeinden sind nur im aktuellen df_h4_processed (tarife) vorhanden:")
    display(gemeinden_only_in_current_df_h4_processed)
else:
    print("Es wurden keine Gemeinden gefunden, die nur im aktuellen df_h4_processed (tarife) vorhanden sind.")

## Schritt 5: Gemeindegrenzen laden

**Ziel:** Polygone aller Schweizer Gemeinden laden, damit Adressen einer Gemeinde zugeordnet werden können (Schritt 9) und die Choropleth-Karte gezeichnet werden kann (Schritt 12).

**Vorgehen:** Die swissBOUNDARIES3D-Datei (swisstopo) wird lokal kopiert und der Layer `tlm_hoheitsgebiet` geladen.

**Was der Code macht:**
- Das GPKG wird lokal zwischengespeichert, weil SQLite-Dateien (wie GPKG) nicht direkt per URL geöffnet werden können
- `engine='fiona'` gibt den korrekten Treiber zum Öffnen von GPKG-Dateien an
- `to_crs('EPSG:4326')` konvertiert das Koordinatensystem von Schweizer LV95 (EPSG:2056, in Metern) zu WGS84 (EPSG:4326, in Grad) – dasselbe System wie PVOUT.tif und die Geocoding-Koordinaten

In [ ]:
import geopandas as gpd
import requests # Importiere Anfragen zum Herunterladen von Dateien
import os

GPKG_LOCAL = '/content/swissboundaries.gpkg'
if not os.path.exists(GPKG_LOCAL):
    print('Lade GPKG von GitHub auf lokalen Speicher...')
    try:
        response = requests.get(GPKG_PATH)
        response.raise_for_status() # Löst einen HTTPError für bad responses (4xx oder 5xx) aus
        with open(GPKG_LOCAL, 'wb') as f:
            f.write(response.content)
        print('Heruntergeladen')
    except requests.exceptions.RequestException as e:
        print(f"Fehler beim Herunterladen des GPKG: {e}")
        raise # Wirft die Ausnahme erneut, um einen kritischen Fehler anzuzeigen
else:
    print('GPKG bereits lokal vorhanden.')

gemeinden_geo   = gpd.read_file(GPKG_LOCAL, layer='tlm_hoheitsgebiet', engine='fiona')
gemeinden_wgs84 = gemeinden_geo.to_crs('EPSG:4326')

print(f'Gemeinden geladen    : {len(gemeinden_wgs84)}')
print(f'Koordinatensystem    : {gemeinden_wgs84.crs}')
gemeinden_wgs84[['bfs_nummer', 'name']].head(3)

## Schritt 6: Adresse eingeben

**Ziel:** Eine beliebige Schweizer Adresse angeben, für die das PV-Potenzial berechnet und mit allen Schweizer Gemeinden verglichen werden soll.

**Vorgehen:** Adresse im Eingabefeld unten eingeben und auf **«Adresse übernehmen»** klicken. Danach Schritte 7–13 einzeln ausführen. Wichtig: nicht den Code mittels "Alles ausführen" aktualisieren.

**Tipps:**
- Zum Vergleich eine sonnige Bergadresse ausprobieren (z.B. Täsch – Südexposition im Mattertal)
- Im flachen Mittelland haben Nachbaradressen oft ähnliche PVOUT-Werte – das ist korrekt (keine Topografie)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Styling
display(HTML("""
<style>
  .pv-card {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    border-radius: 16px; padding: 26px 30px; max-width: 640px;
    box-shadow: 0 8px 32px rgba(0,0,0,.45);
    font-family: 'Segoe UI', Arial, sans-serif; margin: 12px 0;
  }
  .pv-card h3 { color: #f0c040; margin: 0 0 4px 0; font-size: 1.2em; letter-spacing: .5px; }
  .pv-card p  { color: #b0bec5; font-size: .87em; margin: 0; }
  .pv-lbl     { color: #90caf9; font-size: .8em; font-weight: 600;
                text-transform: uppercase; letter-spacing: .7px;
                display: block; margin: 16px 0 6px 0; }
</style>
<div class='pv-card'>
  <h3>📍 PV-Potenzial – Adresseingabe</h3>
  <p>Adresse eingeben und auf «Adresse übernehmen» klicken, dann Schritte 7–13 ausführen.</p>
</div>
"""))

# Eingabefeld
adresse_input = widgets.Text(
    value='Bundesplatz 3, Bern, Schweiz', # Adresse ist hardgecodiert
    placeholder='z.B. Bahnhofstrasse 1, Zürich, Schweiz',
    description='',
    style={'description_width': '0px'},
    layout=widgets.Layout(width='490px', height='40px',
                          border='1.5px solid #1565c0',
                          border_radius='8px', padding='4px 10px')
)

btn_confirm = widgets.Button(
    description='✔  Adresse übernehmen',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='42px',
                          border_radius='8px', margin='0 0 0 10px')
)

# Schnellauswahl-Buttons
beispiele = [
    ('☀️ Lugano',  'Lugano, Schweiz'),
    ('🏔 Täsch',   'Täsch, Schweiz'),
    ('🏙 Zürich',  'Zürich, Schweiz'),
    ('🌉 Basel',   'Basel, Schweiz'),
    ('🏡 Bern',    'Bern, Schweiz'),
]
beispiel_btns = []
for label, addr in beispiele:
    b = widgets.Button(
        description=label, button_style='',
        layout=widgets.Layout(width='115px', height='34px',
                              border_radius='6px', margin='4px 4px 0 0')
    )
    def _make_cb(a):
        def cb(_): adresse_input.value = a
        return cb
    b.on_click(_make_cb(addr))
    beispiel_btns.append(b)

# Status-Ausgabe & Callback
out_status = widgets.Output()
ADRESSE = adresse_input.value   # globale Variable für Folgeschritte

def on_confirm(_):
    global ADRESSE
    ADRESSE = adresse_input.value.strip()
    with out_status:
        out_status.clear_output()
        display(HTML(f"""
        <div style='background:#1b5e20;border-radius:8px;padding:10px 16px;
                    color:#a5d6a7;font-size:.92em;margin-top:8px;
                    font-family:Segoe UI,Arial,sans-serif;'>
          ✅ <strong>Gespeichert:</strong> {ADRESSE}<br>
          <span style='font-size:.84em;color:#81c784;'>
            Jetzt Schritt 7 bis 13 einzeln ausführen.
          </span>
        </div>
        """))

btn_confirm.on_click(on_confirm)

# Layout anzeigen
display(HTML("<span class='pv-lbl'>🔎 Adresse eingeben</span>"))
display(widgets.HBox([adresse_input, btn_confirm]))
display(HTML("<div style='color:#607d8b;font-size:.79em;margin:8px 0 4px 2px;'>"
             "⚡ Schnellauswahl:</div>"))
display(widgets.HBox(beispiel_btns))
display(out_status)


## Schritt 7: Geocoding – Adresse zu Koordinaten

**Ziel:** Die eingegebene Adresse in geografische Koordinaten (Breitengrad, Längengrad) umwandeln.

**Vorgehen:** Zwei Geocoding-Dienste werden nacheinander versucht.

**Was der Code macht:**
1. **swisstopo API (Hauptquelle):** `api3.geo.admin.ch` kennt alle offiziellen Schweizer Adressen aus dem Gebäude- und Wohnungsregister (GWR). Liefert sehr präzise Koordinaten direkt in WGS84 (lat/lon).
2. **Nominatim/OpenStreetMap (Fallback):** Wird verwendet, wenn die swisstopo API keine Ergebnisse liefert – z.B. bei unvollständigen oder nicht offiziellen Adressen.

**Hinweis:** Die swisstopo API kennt nur Schweizer Adressen. Für ausländische Orte greift der Code automatisch auf Nominatim zurück.

In [ ]:
import requests
from geopy.geocoders import Nominatim

def geocodiere_adresse(adresse):
    try:
        url = 'https://api3.geo.admin.ch/rest/services/api/SearchServer'
        params = {'type': 'locations', 'searchText': adresse, 'limit': 1, 'sr': '4326'}
        r = requests.get(url, params=params, timeout=10)
        results = r.json().get('results', [])
        if results:
            attrs = results[0]['attrs']
            return attrs['lat'], attrs['lon'], attrs.get('label', adresse), 'swisstopo'
    except Exception:
        pass
    try:
        gc = Nominatim(user_agent='bina-pv-projekt')
        ort = gc.geocode(adresse)
        if ort:
            return ort.latitude, ort.longitude, ort.address, 'Nominatim'
    except Exception:
        pass
    return None, None, None, None

lat, lon, gefunden, quelle = geocodiere_adresse(ADRESSE)

if not lat:
    print(f'Adresse nicht gefunden: {ADRESSE}')
    print('Tipp: Ort und Schweiz anfügen, z.B. Hauptstrasse 1, Bern, Schweiz')
else:
    print(f'Geocoding ({quelle}): {gefunden}')
    print(f'Koordinaten          : {lat:.5f}N, {lon:.5f}E')

## Schritt 8: PVOUT-Pixelwert an der Adresse ablesen

**Ziel:** Den PVOUT-Wert (kWh/kWp/Jahr) am genauen Standort der eingegebenen Adresse aus dem Raster auslesen.

**Was der Code macht:**
- `rasterio.open(PVOUT_PATH)` öffnet das GeoTIFF
- `rowcol(transform, lon, lat)` berechnet aus den geografischen Koordinaten die Zeile und Spalte im Pixelgitter
- `src.read(1)[r, c]` liest den Pixelwert an dieser Position direkt aus dem Array

**Hinweis zur Genauigkeit:**  
Die Datei hat eine Auflösung von ca. 1 km × 1 km pro Pixel. Das bedeutet: Zwei Adressen im gleichen Quartier können denselben Pixelwert haben – das ist ein Auflösungseffekt der Datenquelle, kein Fehler.
Im flachen Mittelland sind auch Nachbarpixel ähnlich (kaum Topografie). Grosse Abweichungen entstehen in Bergregionen: Nord- vs. Südhang können 30–40 % auseinanderliegen.

In [ ]:
import rasterio
from rasterio.transform import rowcol

with rasterio.open(PVOUT_PATH) as src:
    r, c = rowcol(src.transform, lon, lat)
    pvout_adresse = float(src.read(1)[r, c])

print(f'PVOUT an der Adresse : {pvout_adresse:.0f} kWh/kWp/Jahr')
print('(Pixelaufloesung ca. 1 km x 1 km - Nachbaradressen haben denselben Wert)')

## Schritt 9: Gemeinde der Adresse bestimmen (Spatial Join)

**Ziel:** Herausfinden, zu welcher Gemeinde die eingegebene Adresse gehört.

**Was der Code macht:**
- `Point(lon, lat)` erstellt ein Punkt-Objekt aus den Koordinaten
- `gpd.sjoin(..., predicate='within')` prüft für jeden Punkt, in welchem Gemeindepolygon er liegt
- Das Ergebnis gibt BFS-Nummer und Gemeindename zurück
- Wenn der Punkt ausserhalb aller Polygone liegt (z.B. auf einem See), wird `None` gesetzt

**Warum brauchen wir die Gemeinde?**  
Mit der BFS-Nummer können wir in Schritt 10 den zugehörigen Stromtarif aus `df_h4_processed.csv` nachschlagen.

In [ ]:
from shapely.geometry import Point

punkt   = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs='EPSG:4326')
treffer = gpd.sjoin(punkt, gemeinden_wgs84[['bfs_nummer', 'name', 'geometry']], how='left', predicate='within')

if len(treffer) > 0 and not treffer['name'].isna().all():
    gemeinde_nr   = int(treffer['bfs_nummer'].values[0])
    gemeinde_name = treffer['name'].values[0]
    print(f'Gemeinde             : {gemeinde_name} (BFS-Nr: {gemeinde_nr})')
else:
    gemeinde_nr   = None
    gemeinde_name = 'unbekannt'
    print('Gemeinde konnte nicht ermittelt werden (Koordinate ausserhalb CH-Grenzen?)')

## Schritt 10: Stromtarif nachschlagen und PV-Score berechnen

**Ziel:** Den Stromtarif der ermittelten Gemeinde nachschlagen und den PV-Attraktivitätsscore berechnen.

**Score-Formel:** `pv_score = PVOUT [kWh/kWp/Jahr] × Stromtarif [Rp./kWh]`  
**Einheit:** Rp/kWp/Jahr

**Beispielrechnung:**
- PVOUT = 1150 kWh/kWp/Jahr (typisches Mittelland)
- Tarif = 25 Rp./kWh
- Score = 28 750 Rp/kWp/Jahr
- 5-kWp-Anlage: 28 750 × 5 / 100 = **1437 CHF/Jahr Einsparung**

**Was passiert, wenn kein Tarif gefunden wird?**  
Wenn die Gemeinde nicht in `df_h4_processed.csv` vorhanden ist, wird der Schweizer Durchschnittstarif verwendet und ein Hinweis ausgegeben. Der Score ist dann eine Schätzung.

In [ ]:
if gemeinde_nr is not None:
    tarif_zeile = tarife[tarife['gemeindeNummer'] == gemeinde_nr]
    if len(tarif_zeile) > 0:
        strompreis    = float(tarif_zeile['total'].values[0])
        netzbetreiber = tarif_zeile['netzbetreiber'].values[0]
    else:
        strompreis    = float(tarife['total'].mean())
        netzbetreiber = f'Schweizer Durchschnitt (kein Tarif fuer {gemeinde_name} gefunden)'
else:
    strompreis    = float(tarife['total'].mean())
    netzbetreiber = 'Schweizer Durchschnitt (Gemeinde nicht ermittelt)'

pv_score = pvout_adresse * strompreis

print(f'Netzbetreiber        : {netzbetreiber}')
print(f'Stromtarif H4        : {strompreis:.1f} Rp./kWh')
print(f'PVOUT Standort       : {pvout_adresse:.0f} kWh/kWp/Jahr')
print(f'PV-Score             : {pv_score:.0f} Rp/kWp/Jahr')
print()
print(f'  5-kWp-Anlage spart ca. {pv_score * 5 / 100:.0f} CHF/Jahr')

## Schritt 11: Nationaler Vergleich und Ranking (Qualitätskontrolle 2)

**Ziel:** Den PV-Score für alle Schweizer Gemeinden berechnen, ein Ranking erstellen und den Rang der eingegebenen Gemeinde im nationalen Vergleich zeigen.

**Was der Code macht:**
- Inner Join: Nur Gemeinden mit PVOUT- **und** Tarif-Daten werden berücksichtigt
- `pv_score = pvout_kwh_kwp_year × total` für jede Gemeinde
- Sortierung absteigend → beste Gemeinde = Rang 1

**Plausibilitätsprüfung der Rangliste:**

Die Top-Gemeinden sollten hohen PVOUT **und** hohen Tarif haben. Das sind typischerweise:
- Sonnige Gemeinden im Wallis oder Tessin, wenn dort die Tarife nicht sehr niedrig sind
- Oder Mittelland-Gemeinden mit teurem Strom, wenn ihre Sonneneinstrahlung ausreicht

Die Tabelle zeigt alle relevanten Spalten: `pvout_kwh_kwp_year × total = pv_score` – so kann man die Berechnung direkt nachvollziehen.

In [ ]:
ranking = pvout.merge(
    tarife[['gemeindeNummer', 'total', 'netzbetreiber']],
    left_on='gemeinde_nr',
    right_on='gemeindeNummer',
    how='inner'
)
ranking['pv_score'] = ranking['pvout_kwh_kwp_year'] * ranking['total']
ranking = ranking.sort_values('pv_score', ascending=False).reset_index(drop=True)
ranking['rang'] = ranking.index + 1
total_gemeinden = len(ranking)

print(f'Gemeinden im Ranking : {total_gemeinden}')
print()

cols_show = ['rang', 'gemeinde_name', 'kanton_name', 'pvout_kwh_kwp_year', 'total', 'pv_score']
print('=== Top 10 – höchstes PV-Potenzial ===')
print(ranking[cols_show].head(10).to_string(index=False))

print()
print('=== Bottom 10 – niedrigstes PV-Potenzial ===')
print(ranking[cols_show].tail(10).to_string(index=False))

In [ ]:
mein_rang = ranking[ranking['gemeinde_nr'] == gemeinde_nr]['rang'].values

if len(mein_rang) > 0:
    rang = int(mein_rang[0])
    print(f'{gemeinde_name}: Rang {rang} von {total_gemeinden}')
    print(f'Besser als {100 * (1 - rang / total_gemeinden):.0f}% aller Schweizer Gemeinden')
else:
    rang = None
    print(f'{gemeinde_name}: nicht im Ranking (keine Tarif-Daten verfügbar)')
    print('Die Karte und Zusammenfassung zeigen den Score der Adresse ohne nationalen Rang.')

## Schritt 12: Interaktive Choropleth-Karte (Folium)

**Ziel:** Alle Schweizer Gemeinden auf einer interaktiven Karte nach ihrem PV-Score einfärben und die eingegebene Adresse markieren.

**Was der Code macht:**
- `karte_daten` verknüpft die Gemeindegeometrie mit den Score-Daten. Wichtig: Nur die benötigten Spalten werden übernommen – das verhindert einen Serialisierungsfehler, der entsteht, wenn Datum-Spalten aus dem GPKG in das Choropleth-Format exportiert werden.
- `folium.Choropleth(fill_color='RdYlGn')` färbt die Gemeinden: Rot = tiefer Score, Grün = hoher Score
- `nan_fill_color='lightgrey'` zeigt Gemeinden ohne Tarif-Daten grau
- `folium.Marker` setzt einen blauen Hausmarker an die eingegebene Adresse mit Score und Rang als Tooltip

**Interaktion:** Karte zoomen, verschieben, Marker anklicken für Details.

In [ ]:
import folium

# Nur benötigte Spalten übernehmen – vermeidet Timestamp-Serialisierungsfehler
karte_daten = gemeinden_wgs84[['bfs_nummer', 'name', 'geometry']].merge(
    ranking[['gemeinde_nr', 'pv_score', 'rang']],
    left_on='bfs_nummer',
    right_on='gemeinde_nr',
    how='left'
)
karte_daten = karte_daten[['bfs_nummer', 'name', 'pv_score', 'rang', 'geometry']].copy()

m = folium.Map(location=[46.8, 8.2], zoom_start=8, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=karte_daten.__geo_interface__,
    data=karte_daten,
    columns=['bfs_nummer', 'pv_score'],
    key_on='feature.properties.bfs_nummer',
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='PV-Score [Rp/kWp/Jahr]',
    nan_fill_color='lightgrey'
).add_to(m)

rang_text = f'Rang {rang} von {total_gemeinden}' if rang is not None else 'kein Rang (keine Tarif-Daten)'
folium.Marker(
    location=[lat, lon],
    tooltip=f'{ADRESSE} | Score: {pv_score:.0f} Rp/kWp/Jahr | {rang_text}',
    icon=folium.Icon(color='blue', icon='home')
).add_to(m)

m

## Schritt 13: Zusammenfassung

**Ziel:** Alle Ergebnisse für die eingegebene Adresse visuell aufbereitet zusammenfassen.

**Was der Code macht:**
- Zeigt PVOUT, Tarif und Score als KPI-Kacheln
- Visualisiert den nationalen Rang als Fortschrittsbalken
- Berechnet die Jahreseinsparung für drei verschiedene Anlagengrössen


In [ ]:
from IPython.display import display, HTML

# Rang-Metriken
if rang is not None:
    pct        = round(100 * (1 - rang / total_gemeinden))
    rang_text  = f'Rang {rang} von {total_gemeinden}'
    rang_badge = f'Besser als {pct} % aller Schweizer Gemeinden'
    bar_w      = pct
    bar_col    = '#4caf50' if pct >= 66 else '#ff9800' if pct >= 33 else '#f44336'
    schaetzung = ''
else:
    rang_text  = 'nicht verfügbar'
    rang_badge = 'Keine Tarif-Daten für diese Gemeinde'
    bar_w      = 0
    bar_col    = '#607d8b'
    schaetzung = ' (Schätzung)'

# Anlagen-Tabelle
anlagen_rows = ''.join(
    f"""<tr>
      <td style='padding:9px 16px;color:#b0bec5;'>{ico} {kw} kWp</td>
      <td style='padding:9px 16px;color:#a5d6a7;font-weight:700;text-align:right;'>
        {pv_score * kw / 100:,.0f} CHF / Jahr{schaetzung}</td>
    </tr>"""
    for kw, ico in [(3, '🏡'), (5, '🏠'), (10, '🏢')]
)

# HTML-Ausgabe
html = f"""
<style>
  .pvd {{
    background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
    border-radius:18px; padding:30px 36px; max-width:700px;
    box-shadow:0 10px 40px rgba(0,0,0,.55);
    font-family:'Segoe UI',Arial,sans-serif; color:#eceff1; margin:16px 0;
  }}
  .pvd h2  {{ margin:0 0 3px 0; color:#f0c040; font-size:1.3em; letter-spacing:.5px; }}
  .pvd .sub{{ color:#90a4ae; font-size:.87em; margin:0 0 22px 0; }}
  .pvd-addr{{
    background:rgba(255,255,255,.07); border-radius:10px;
    padding:10px 16px; color:#cfd8dc; font-size:.9em;
    border-left:3px solid #f0c040; margin-bottom:20px;
  }}
  .pvd-grid{{
    display:grid; grid-template-columns:repeat(3,1fr); gap:14px; margin-bottom:22px;
  }}
  .pvd-kpi {{
    background:rgba(255,255,255,.07); border-radius:12px;
    padding:14px 12px; text-align:center;
  }}
  .pvd-kpi .val  {{ font-size:1.6em; font-weight:700; color:#80cbc4; line-height:1.1; }}
  .pvd-kpi .unit {{ font-size:.71em; color:#607d8b; display:block; margin-top:2px; }}
  .pvd-kpi .lbl  {{ font-size:.73em; color:#90a4ae; margin-top:6px;
                    text-transform:uppercase; letter-spacing:.6px; }}
  .pvd-sec       {{ margin-top:20px; }}
  .pvd-sec h4    {{ color:#90caf9; font-size:.8em; text-transform:uppercase;
                    letter-spacing:.8px; margin:0 0 8px 0; }}
  .pvd-bar-bg    {{ background:rgba(255,255,255,.1); border-radius:8px;
                    height:18px; overflow:hidden; }}
  .pvd-bar-fg    {{ height:18px; border-radius:8px; background:{bar_col}; width:{bar_w}%; }}
  .pvd-bar-lbl   {{ color:#b0bec5; font-size:.82em; margin-top:6px; }}
  .pvd-tbl       {{ width:100%; border-collapse:collapse; margin-top:8px; }}
  .pvd-tbl th    {{ color:#90caf9; font-size:.77em; text-transform:uppercase;
                    letter-spacing:.6px; padding:6px 16px; text-align:left;
                    border-bottom:1px solid rgba(255,255,255,.1); }}
  .pvd-tbl th:last-child {{ text-align:right; }}
  .pvd-tbl tr:nth-child(even) td {{ background:rgba(255,255,255,.04); }}
  .pvd-foot {{ margin-top:18px; color:#546e7a; font-size:.77em; }}
</style>

<div class='pvd'>
  <h2>☀️ PV-Potenzial – Ergebnisübersicht</h2>
  <p class='sub'>Nationaler Vergleich aller Schweizer Gemeinden</p>

  <div class='pvd-addr'>
    📍 <strong>{ADRESSE}</strong>&nbsp; · &nbsp;{gemeinde_name}
  </div>

  <div class='pvd-grid'>
    <div class='pvd-kpi'>
      <div class='val'>{pvout_adresse:.0f}</div>
      <span class='unit'>kWh / kWp / Jahr</span>
      <div class='lbl'>PVOUT Standort</div>
    </div>
    <div class='pvd-kpi'>
      <div class='val'>{strompreis:.1f}</div>
      <span class='unit'>Rp. / kWh</span>
      <div class='lbl'>Stromtarif H4</div>
    </div>
    <div class='pvd-kpi'>
      <div class='val'>{pv_score:.0f}</div>
      <span class='unit'>Rp / kWp / Jahr</span>
      <div class='lbl'>PV-Score</div>
    </div>
  </div>

  <div class='pvd-sec'>
    <h4>🏆 Nationaler Rang</h4>
    <div class='pvd-bar-bg'><div class='pvd-bar-fg'></div></div>
    <div class='pvd-bar-lbl'>
      <strong style='color:#eceff1;'>{rang_text}</strong>&nbsp; · &nbsp;{rang_badge}
    </div>
  </div>

  <div class='pvd-sec'>
    <h4>💡 Geschätzte Jahreseinsparung nach Anlagengrösse</h4>
    <table class='pvd-tbl'>
      <tr><th>Anlage</th><th>Einsparung / Jahr</th></tr>
      {anlagen_rows}
    </table>
  </div>

  <div class='pvd-foot'>Netzbetreiber: {netzbetreiber}</div>
</div>
"""

display(HTML(html))
